# Detección de Fraude en Transacciones con Tarjeta de Crédito

**Dataset**: Credit Card Fraud Detection (Kaggle - mlg-ulb/creditcardfraud)

**Objetivo**: Análisis exploratorio + modelo de clasificación (XGBoost) para detectar transacciones fraudulentas, con foco en métricas adecuadas para datasets desbalanceados.

**Estructura**:
1. Carga y exploración inicial
2. Análisis del desbalance de clases
3. EDA de variables (Time, Amount, V1-V28)
4. Correlaciones
5. Preprocesamiento
6. Modelado con XGBoost
7. Evaluación: matriz de confusión, ROC-AUC, Precision-Recall
8. Exportar modelo y scaler

## 1. Carga y exploración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score, ConfusionMatrixDisplay
)

import xgboost as xgb
import joblib

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Descarga el dataset desde https://www.kaggle.com/mlg-ulb/creditcardfraud
# y colócalo en ../data/creditcard.csv
df = pd.read_csv('../data/creditcard.csv')

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum().sum(), 'valores nulos en total')

# Estadísticas descriptivas
df.describe()

## 2. Análisis del desbalance de clases

Este es el punto MÁS importante del proyecto: el dataset está extremadamente desbalanceado (~0.17% de fraudes). Esto determina toda la estrategia de modelado y evaluación posterior — **la accuracy NO es una métrica útil aquí**.

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('Conteo de clases:')
print(class_counts)
print('\nPorcentaje:')
print(class_pct.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='Class', data=df, ax=axes[0])
axes[0].set_title('Distribución de clases (escala normal)')
axes[0].set_xticklabels(['Normal (0)', 'Fraude (1)'])

sns.countplot(x='Class', data=df, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Distribución de clases (escala logarítmica)')
axes[1].set_xticklabels(['Normal (0)', 'Fraude (1)'])

plt.tight_layout()
plt.show()

print(f'\nRatio fraude/normal: 1 fraude por cada {class_counts[0] // class_counts[1]} transacciones normales')

## 3. EDA de variables: Time y Amount

`Time` representa los segundos transcurridos desde la primera transacción del dataset. `Amount` es el monto de la transacción. Las variables V1-V28 ya vienen anonimizadas/transformadas con PCA por motivos de privacidad.

In [ ]:
# Convertir Time a horas para mejor interpretación
df['Hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(data=df, x='Hour', hue='Class', bins=24, multiple='stack', ax=axes[0])
axes[0].set_title('Distribución de transacciones por hora')

# Solo fraudes para ver el patrón horario sin que lo tape el volumen normal
sns.histplot(data=df[df['Class']==1], x='Hour', bins=24, color='red', ax=axes[1])
axes[1].set_title('Distribución horaria SOLO de fraudes')

plt.tight_layout()
plt.show()

In [ ]:
# Distribución del monto (Amount) por clase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0])
axes[0].set_title('Distribución de Amount por clase')
axes[0].set_xticklabels(['Normal', 'Fraude'])

# Acotar el eje Y porque hay outliers extremos en transacciones normales
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[1])
axes[1].set_ylim(0, 500)
axes[1].set_title('Distribución de Amount por clase (zoom 0-500)')
axes[1].set_xticklabels(['Normal', 'Fraude'])

plt.tight_layout()
plt.show()

print('Estadísticas de Amount por clase:')
print(df.groupby('Class')['Amount'].describe())

## 4. Correlaciones de V1-V28 con la clase

Identificamos qué variables (componentes PCA) tienen mayor relación con el fraude. Esto ayuda a entender qué "impulsa" las predicciones del modelo más adelante.

In [ ]:
corr_with_class = df.corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(8, 10))
colors = ['crimson' if v < 0 else 'steelblue' for v in corr_with_class.values]
plt.barh(corr_with_class.index, corr_with_class.values, color=colors)
plt.title('Correlación de cada variable con Class (fraude)')
plt.xlabel('Coeficiente de correlación')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('Top 5 correlaciones positivas:')
print(corr_with_class.sort_values(ascending=False).head())
print('\nTop 5 correlaciones negativas:')
print(corr_with_class.sort_values().head())

In [ ]:
# Visualizar las 4 variables más correlacionadas (positiva y negativamente) con Class
top_features = corr_with_class.abs().sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flatten(), top_features):
    sns.kdeplot(data=df, x=feat, hue='Class', fill=True, common_norm=False, ax=ax)
    ax.set_title(f'Distribución de {feat} por clase')

plt.tight_layout()
plt.show()

## 5. Preprocesamiento

- Escalamos `Amount` y `Time` (las V1-V28 ya vienen escaladas por el PCA original)
- Split estratificado train/test para preservar la proporción de clases en ambos conjuntos
- Eliminamos la columna `Hour` creada para EDA (no la usaremos en el modelo)

In [ ]:
df_model = df.drop(columns=['Hour'])

X = df_model.drop(columns=['Class'])
y = df_model['Class']

# Escalar Time y Amount
scaler = StandardScaler()
X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

# Split estratificado: mantiene la proporción ~0.17% de fraude en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape}, fraudes: {y_train.sum()} ({y_train.mean()*100:.3f}%)')
print(f'Test:  {X_test.shape}, fraudes: {y_test.sum()} ({y_test.mean()*100:.3f}%)')

## 6. Modelado con XGBoost

Usamos `scale_pos_weight` para manejar el desbalance: le decimos al modelo que penalice más los errores en la clase minoritaria (fraude), en lugar de hacer oversampling/undersampling. Esto es eficiente y evita duplicar/perder información.

In [ ]:
# scale_pos_weight = ratio negativos/positivos en el set de entrenamiento
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',  # área bajo la curva precision-recall, más adecuada que logloss aquí
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

## 7. Evaluación

### 7.1 Predicciones y matriz de confusión

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Fraude'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Matriz de Confusión')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Negativos (normales bien clasificados): {tn}')
print(f'Falsos Positivos (normales marcados como fraude):  {fp}')
print(f'Falsos Negativos (fraudes NO detectados):          {fn}  <- el más crítico de minimizar')
print(f'Verdaderos Positivos (fraudes detectados):         {tp}')

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude'], digits=4))

### 7.2 Curva ROC y AUC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc:.4f})', color='steelblue', linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Clasificador aleatorio')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos (Recall)')
plt.title('Curva ROC')
plt.legend()
plt.show()

print(f'ROC-AUC: {roc_auc:.4f}')

### 7.3 Curva Precision-Recall

**Importante**: en datasets tan desbalanceados, la curva Precision-Recall es más informativa que la ROC, porque la ROC puede verse "optimista" cuando la clase negativa domina ampliamente.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

plt.figure(figsize=(7, 6))
plt.plot(recall, precision, color='darkorange', linewidth=2,
         label=f'XGBoost (AP = {avg_precision:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall')
plt.legend()
plt.show()

print(f'Average Precision (AP): {avg_precision:.4f}')

In [ ]:
# Análisis del umbral: ¿cómo cambian precision/recall según el threshold elegido?
# Por defecto el threshold es 0.5, pero en fraude suele bajarse para priorizar recall
thresholds_to_test = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9]

print(f'{"Threshold":<10} {"Precision":<10} {"Recall":<10} {"Fraudes detectados":<20}')
for t in thresholds_to_test:
    y_pred_t = (y_proba >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    prec = tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0
    rec = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0
    print(f'{t:<10} {prec:<10.4f} {rec:<10.4f} {tp_t}/{tp_t+fn_t}')

### 7.4 Importancia de features

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
importances.head(15).plot(kind='barh')
plt.title('Top 15 features más importantes (XGBoost)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Exportar modelo y scaler

Guardamos los artefactos para usarlos en la API de FastAPI.

In [ ]:
joblib.dump(model, '../backend/model.pkl')
joblib.dump(scaler, '../backend/scaler.pkl')
joblib.dump(list(X.columns), '../backend/feature_names.pkl')

print('Modelo, scaler y nombres de features guardados en /backend')

## Conclusiones

- El dataset presenta un desbalance extremo (~0.17% fraude), por lo que la **accuracy no es una métrica válida**.
- Se priorizó **recall** (detectar la mayor cantidad de fraudes posible) usando `scale_pos_weight` en XGBoost en vez de oversampling, para no introducir datos sintéticos.
- La curva **Precision-Recall** y el **AUC-PR** son las métricas más representativas del desempeño real del modelo en este contexto.
- El análisis de threshold muestra el trade-off entre precision y recall: bajar el umbral aumenta la detección de fraudes (recall) a costa de más falsos positivos (revisión manual).
- Próximos pasos: exponer el modelo vía API (FastAPI) y construir un dashboard en Next.js que permita simular transacciones y visualizar estas métricas.